# CPU, CUDA et AMP : placer les calculs et compter la mémoire

Déplacer une boucle comprise avant d’en mesurer le coût sur une machine.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#cuda-amp-memoire).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La boucle est correcte sur CPU. Nous pouvons maintenant examiner son emplacement et sa précision, sans modifier en même temps les données, la tâche et le modèle.

Nous reprenons les trois extraits fictifs et une Linear(2,2). Le choix device indique où stocker les paramètres et exécuter leurs opérations : CPU ou GPU CUDA compatible.

| Objet | Forme | Type |
| --- | --- | --- |
| Entrées | (3,2) | float32 |
| Cibles | (3,) | int64 |
| Poids et biais | (2,2) et (2,) | float32 |

Ces objets doivent résider sur des emplacements compatibles. Le déplacement n’invente aucune nouvelle valeur ni annotation.

La précision mixte automatique, AMP, choisit le format de certaines opérations. autocast encadre ce choix ; GradScaler met à l’échelle la perte pour protéger certains petits gradients.

Le programme active AMP seulement pour CUDA. Sur la machine de vérification, seul le chemin CPU a été exécuté : nous ne disposons donc d’aucune mesure de vitesse ou de mémoire GPU.

## Démonstration 1 · Identifier ce qui change dans la boucle

Le chemin CPU reste directement exécutable.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Fixer les conditions du petit calcul

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
torch.manual_seed(17)


### Calculer l’état suivant

is_available teste la disponibilité CUDA dans l’environnement actuel. .to(device) renvoie les objets placés sur le même emplacement choisi.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modele = nn.Linear(2, 2).to(device)
entrees = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]]).to(device)
cibles = torch.tensor([0, 1, 0], dtype=torch.long).to(device)


In [6]:
observer('device', device)
observer('entrees', entrees)
observer('cibles', cibles)

entrees forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Choisir les paramètres et leur mise à jour

CrossEntropyLoss reçoit toujours des scores flottants et un indice de classe par extrait. Changer de device ne change pas ce contrat.

In [7]:
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)


### Calculer l’état suivant

amp_active est un booléen. Quand il est faux, autocast et le scaler ne modifient pas le calcul CPU de référence.

In [8]:
amp_active = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_active)


In [9]:
observer('amp_active', amp_active)
observer('scaler', scaler)

amp_active = False


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
optimiseur.zero_grad()


In [11]:
observer_entrainement({'device': device, 'modele': modele, 'entrees': entrees, 'cibles': cibles, 'critere': critere, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler}, montrer_poids=False)

modele.weight.grad = None
modele.bias.grad = None


### Appliquer le contexte explicite

autocast couvre le calcul des scores et de la perte. La sortie conserve les trois extraits et les deux classes.

In [12]:
with torch.amp.autocast(device_type=device.type, enabled=amp_active):
    scores = modele(entrees)
    perte = critere(scores, cibles)


### Calculer les gradients

Le scaler encadre backward et step, puis adapte son échelle. Ce protocole remplace le couple backward/step brut lorsque la branche AMP est active.

In [13]:
scaler.scale(perte).backward()


In [14]:
observer_entrainement({'device': device, 'modele': modele, 'entrees': entrees, 'cibles': cibles, 'critere': critere, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler}, montrer_poids=False)

modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.023643290624022484, 0.7529823184013367], [-0.023643292486667633, -0.7529823780059814]]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.009377002716064453, -0.009377047419548035]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [15]:
scaler.step(optimiseur)


In [16]:
observer_entrainement({'device': device, 'modele': modele, 'entrees': entrees, 'cibles': cibles, 'critere': critere, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler}, montrer_poids=True)

modele.weight forme (2, 2) type torch.float32 requires_grad True
  valeurs : [[-0.09536128491163254, -0.025645777583122253], [0.4693499803543091, -0.45663899183273315]]
modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.023643290624022484, 0.7529823184013367], [-0.023643292486667633, -0.7529823780059814]]
modele.bias forme (2,) type torch.float32 requires_grad True
  valeurs : [-0.6665782332420349, 0.07079435139894485]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.009377002716064453, -0.009377047419548035]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [17]:
scaler.update()


### Calculer l’état suivant

numel compte les valeurs ; element_size donne leurs octets. Quatre poids et deux biais float32 donnent (4+2)×4=24 octets.

In [18]:
octets_poids = sum(p.numel() * p.element_size() for p in modele.parameters())


In [19]:
observer('octets_poids', octets_poids)

octets_poids = 24


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [20]:
print("device :", device.type, "AMP active :", amp_active)
print("formes :", tuple(entrees.shape), tuple(scores.shape))
print("octets des parametres :", octets_poids)


device : cpu AMP active : False
formes : (3, 2) (3, 2)
octets des parametres : 24


### Calculer l’état suivant

La synchronisation attend les travaux CUDA. memory_allocated rapporte les allocations suivies par PyTorch ; ce chiffre n’est pas la mémoire totale du processus.

In [21]:
if device.type == "cuda":
    torch.cuda.synchronize(device)
    print("octets CUDA alloues :", torch.cuda.memory_allocated(device))

Le programme distingue les paramètres comptés sur CPU et la branche GPU à mesurer sur un matériel compatible.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Dans ce notebook vérifié sur CPU, quel placement doit produire le choix avec fallback ?

In [22]:
exo_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
exo_type = ...  # Remplacer les points par votre réponse.
if exo_type is Ellipsis or exo_type is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_type == exo_device.type
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

CUDA_VISIBLE_DEVICES est vide dans l’environnement du notebook. Nous n’inventons pas une exécution GPU.

</details>

### Exercice 2

Combien d’octets logiques occupent vingt poids float32 ?

In [23]:
exo_poids = torch.zeros(5,4,dtype=torch.float32)
exo_octets = ...  # Remplacer les points par votre réponse.
if exo_octets is Ellipsis or exo_octets is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_octets == exo_poids.numel()*exo_poids.element_size()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Comptez les valeurs puis les quatre octets par valeur. Ce n’est pas la mémoire totale d’un entraînement.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
exo_type = "cpu"
if exo_type is Ellipsis or exo_type is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_type == exo_device.type
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_poids = torch.zeros(5,4,dtype=torch.float32)
exo_octets = 80
if exo_octets is Ellipsis or exo_octets is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_octets == exo_poids.numel()*exo_poids.element_size()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [24]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()